# Module 01 AI Workbook — solutions and explanation

> Try the exercises yourself before reading this. The value is in predicting the
> bug and writing the harness, not in seeing the fixed code.

## The adversarial bug

[../adversarial_vector_add_buggy.cu](./adversarial_vector_add_buggy.cu) launches
the kernel and then immediately reads `c` on the host:

```cpp
addVectorsInto<<<blocks, threads>>>(c, a, b, N);
checkElementsAre(7, c, N);        // <-- reads results the GPU may not have written yet
```

A CUDA kernel launch is **asynchronous**: it returns control to the CPU right
away, before the GPU has necessarily finished (or even started) the work. So the
host can read `c` while the kernel is still running.

**Why it sometimes "passes":** with unified memory the check may happen to run
after the GPU finished, or the pages may already hold the right values by luck.
That is what makes the bug dangerous — a test that runs once can pass, and the
failure only appears intermittently or under load. It produces **no compiler
error**.

**The fix** is a single line: synchronize before reading.

```cpp
addVectorsInto<<<blocks, threads>>>(c, a, b, N);
cudaGetLastError();          // also catch launch errors
cudaDeviceSynchronize();     // wait for the GPU before the host reads c
checkElementsAre(7, c, N);
```

The complete fixed program, with an added CPU reference and a PASS/FAIL gate, is
[adversarial_vector_add_fixed.cu](solutions/adversarial_vector_add_fixed.cu).

## The verification harness

[verify_vector_add_solution.cu](solutions/verify_vector_add_solution.cu) is the completed
Phase-4 harness. The parts you were meant to fill in:

1. A launch configuration (256 threads/block; blocks cover `N`).
2. The kernel launch.
3. `cudaGetLastError()` to catch launch failures.
4. `cudaDeviceSynchronize()` so the host waits for the GPU.

It uses `N = 1'000'003`, which is **not** a multiple of the block size — that is
where a dropped bounds check or off-by-one grid math would show up. The
grid-stride kernel with its `i < N` guard handles the partial tile correctly.

## Why this maps to the 5-phase loop

- **SPECIFY / PREDICT:** you should have predicted "missing synchronization"
  from reading the kernel launch, before running anything.
- **VERIFY:** the harness you wrote — not the AI's own check — is what turns an
  intermittent race into a reliable, repeatable `FAIL` you can act on.
- **DIAGNOSE:** vector add is memory-bound (3 floats moved per add, ~1 flop), so
  its speed is set by memory bandwidth, not compute. That is the honest ceiling.

## The point

An AI *may* spot the missing `cudaDeviceSynchronize()` if you ask it to look for
one, but it cannot be relied on to, and it will not be the one held responsible
when the intermittent bug reaches production. The reliable defense is the process
in this workbook: predict the risk, then prove it with a harness you understand.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!nvcc -O3 -arch=native solutions/adversarial_vector_add_fixed.cu -o sol0 && ./sol0

In [ ]:
!nvcc -O3 -arch=native solutions/verify_vector_add_solution.cu -o sol1 && ./sol1